# QTagger+ — Full Technical Notebook
## Quantum Kernel (QSVM) and Variational Quantum Circuit (VQC) Classification of Malware, with Mathematical Derivations

**This notebook contains every piece of code used across the Week 4 experiments, with the underlying mathematics explained alongside each method, and the actual results obtained when this code was run in the working session.**

**Note on cell outputs:** this notebook's cells are pre-populated with the real results from the executions performed during the session (not fabricated) — the notebook itself has **not** been re-executed top-to-bottom here, so `In [ ]` counters are blank. Re-running any cell against `MalMem2022_SMOTE.csv` will reproduce these numbers (small floating-point/timing variation expected from hardware differences).

**Dataset:** `MalMem2022_SMOTE.csv` — 117,192 rows, 55 numeric features, 4-class balanced (Benign / Ransomware / Spyware / Trojan), SMOTE-augmented.

**Contents:**
1. Data loading & stratified sampling
2. Classical preprocessing pipeline (math + code)
3. Quantum feature encoding (math)
4. QSVM — fidelity quantum kernel (math + code + results)
5. VQC — variational quantum circuit (math + code + results)
6. Qubit-capacity sweep (math of the diagnostic + results)
7. Classical baselines & reverification (code + results)
8. Grid extension q∈{8,12}, n∈{250,500,1000} (results)
9. Literature-informed pipeline v2: LDA, data re-uploading, Nyström (math + code + results)
10. Optimizer isolation test & final consolidated comparison


---
## 1. Data Loading & Stratified Sampling

We sample a balanced subset of size $n_{total}$ (equal count per class) from the full 117,192-row dataset, then take a stratified train/test split. Stratification matters here because with only 4 classes and sometimes as few as 50 samples per class (at $n_{total}=200$), a non-stratified split could easily leave a class under-represented in the test set, corrupting F1 macro estimates.


In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

def load_and_split(n_total, seed=42, path='/mnt/user-data/uploads/MalMem2022_SMOTE.csv'):
    df = pd.read_csv(path)
    per_class = n_total // 4
    parts = [g.sample(n=per_class, random_state=seed) for _, g in df.groupby('Label')]
    sub = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    y = sub['Label'].values
    X = sub.drop(columns=['Label'])
    return X, y

X, y = load_and_split(200)
print(X.shape, "features:", X.shape[1])
print(pd.Series(y).value_counts())

(200, 55) features: 55
Ransomware    50
Spyware       50
Trojan        50
Benign        50
Name: count, dtype: int64

---
## 2. Classical Preprocessing Pipeline (v1)

Applied identically across Day 23 and the q=8/12 grid extension. Every fitted transform below is fit **only on the training split**, then applied to test — this is what makes the pipeline leakage-safe.

### 2.1 Variance filter
Drop feature $j$ if $\mathrm{Var}(x_j) \le \epsilon$ (here $\epsilon = 10^{-8}$), computed over the training rows only. Removes constant/near-constant columns that carry no discriminative signal and would otherwise destabilize the correlation and scaling steps.

### 2.2 Correlation filter (leakage-safe)
For the surviving features, compute the training-set correlation matrix
$$ \rho_{jk} = \frac{\mathrm{Cov}(x_j, x_k)}{\sigma_j \sigma_k} $$
and drop one feature from any pair with $|\rho_{jk}| > 0.95$. This is done using the **upper triangular** part of the matrix only, so each redundant pair is resolved once, not twice.

### 2.3 Signed log transform
$$ \tilde{x}_{ij} = \mathrm{sign}(x_{ij}) \cdot \log(1 + |x_{ij}|) $$
Compresses heavy-tailed / skewed feature distributions (common in malware static-analysis features like entropy, byte-histogram counts) while preserving sign, without needing to fit any parameters.

### 2.4 Standardization
$$ z_{ij} = \frac{\tilde{x}_{ij} - \mu_j}{\sigma_j}, \quad \mu_j, \sigma_j \text{ computed on train only} $$

### 2.5 PCA — dimensionality reduction to qubit count
PCA finds the orthogonal directions of maximum variance. Formally, given the (train-only) covariance matrix $\Sigma = \frac{1}{n}Z^\top Z$, we solve the eigenproblem
$$ \Sigma v_k = \lambda_k v_k, \quad \lambda_1 \ge \lambda_2 \ge \dots $$
and project onto the top $q$ (= n_qubits) eigenvectors: $p_i = V_q^\top z_i \in \mathbb{R}^q$. The **explained variance ratio** reported throughout this project is $\sum_{k=1}^{q} \lambda_k / \sum_{k} \lambda_k$ — how much of the original spread survives compression to $q$ dimensions. This is an *unsupervised* criterion: it has no knowledge of the class labels, which becomes relevant in Section 9.

### 2.6 Angle-range scaling
$$ a_{ik} = \pi \cdot \frac{p_{ik} - \min_i(p_{ik})}{\max_i(p_{ik}) - \min_i(p_{ik})} \in [0, \pi] $$
PCA outputs are unbounded reals; quantum rotation gates need bounded angles. Mapping to $[0,\pi]$ (rather than, say, $[0, 2\pi]$) avoids redundant encodings, since $R_Y(\theta)$ and $R_Y(\theta + 2\pi)$ produce the same state but $R_Y(\theta)$ and $R_Y(\pi - \theta)$ do not.


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

def preprocess_pipeline(X, y, n_components, variance_thresh=1e-8, corr_thresh=0.95, seed=42):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    variances = Xtr.var()
    keep_var = variances[variances > variance_thresh].index
    Xtr, Xte = Xtr[keep_var], Xte[keep_var]
    corr = Xtr.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_corr = [c for c in upper.columns if any(upper[c] > corr_thresh)]
    Xtr = Xtr.drop(columns=drop_corr); Xte = Xte.drop(columns=drop_corr)
    Xtr = np.sign(Xtr) * np.log1p(np.abs(Xtr))
    Xte = np.sign(Xte) * np.log1p(np.abs(Xte))
    ss = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = ss.transform(Xtr), ss.transform(Xte)
    n_components = min(n_components, Xtr_s.shape[1], Xtr_s.shape[0])
    pca = PCA(n_components=n_components, random_state=seed).fit(Xtr_s)
    Xtr_p, Xte_p = pca.transform(Xtr_s), pca.transform(Xte_s)
    evr = pca.explained_variance_ratio_.sum()
    mm = MinMaxScaler(feature_range=(0, np.pi)).fit(Xtr_p)
    Xtr_a, Xte_a = mm.transform(Xtr_p), mm.transform(Xte_p)
    return Xtr_a, Xte_a, ytr, yte, evr, Xtr.shape[1]

Xtr, Xte, ytr, yte, evr, nfeat = preprocess_pipeline(X, y, n_components=4)
print(f"features after filtering: {nfeat}")
print(f"PCA(4) explained variance ratio: {evr:.4f}")
print(f"train shape: {Xtr.shape}, test shape: {Xte.shape}")
print(f"angle range check: min={Xtr.min():.3f}, max={Xtr.max():.3f}")

features after filtering: 24
PCA(4) explained variance ratio: 0.8036
train shape: (140, 4), test shape: (60, 4)
angle range check: min=0.000, max=3.142

---
## 3. Quantum Feature Encoding

A classical vector $z \in \mathbb{R}^q$ (here, PCA output mapped to $[0,\pi]$) is embedded into an $q$-qubit quantum state via **angle embedding**:
$$ |\phi(z)\rangle = \bigotimes_{k=1}^{q} R_Y(z_k)|0\rangle, \qquad R_Y(\theta) = \begin{pmatrix} \cos(\theta/2) & -\sin(\theta/2) \\ \sin(\theta/2) & \cos(\theta/2) \end{pmatrix} $$

Each classical feature controls the rotation angle of one qubit. This alone (a *product state*, no entanglement) is not more expressive than a classical feature map — the power of a quantum feature map comes from **entangling gates** applied on top of the rotations, which correlate qubits in ways with no simple classical analogue and expand the effective feature space to the full $2^q$-dimensional Hilbert space.

This is the encoding used for every QSVM/VQC run in Sections 4-5 (v1). Section 9 modifies it to a *repeated* (data re-uploading) version.


---
## 4. QSVM — Quantum Kernel Support Vector Machine

### 4.1 The fidelity kernel
Given two classical samples $z_i, z_j$, encode both into quantum states $|\phi(z_i)\rangle$, $|\phi(z_j)\rangle$ and define their similarity as the **state fidelity**:
$$ K(z_i, z_j) = |\langle \phi(z_i) | \phi(z_j) \rangle|^2 $$

This is computed on hardware/simulator by applying $U_\phi(z_i)$ then $U_\phi(z_j)^\dagger$ to $|0\rangle^{\otimes q}$ and reading the probability of measuring the all-zeros state:
$$ K(z_i, z_j) = |\langle 0 | U_\phi(z_j)^\dagger U_\phi(z_i) | 0 \rangle|^2 = P(\text{measure } 0^q) $$

This is exactly what the `kernel_circuit` below computes: it applies the embedding for $x_1$, the *adjoint* embedding for $x_2$, and reads off `probs()[0]` (the probability of the all-zero bitstring).

$K$ is symmetric ($K(z_i,z_j)=K(z_j,z_i)$) and positive semi-definite by construction (it's an inner-product-squared), so it is a valid Mercer kernel and can be dropped directly into a standard kernel SVM.

### 4.2 Computational cost
Building the full $N \times N$ kernel matrix requires $O(N^2)$ circuit evaluations. This is the reason QSVM runs in this project were capped at 80 training / 40 test samples per run regardless of the nominal `n_total` — at $n=1000$ (800 train) a full kernel matrix would need ~320,000 circuit evaluations for the train block alone, roughly 26 minutes at this project's simulator throughput (~3.3ms/circuit at 8 qubits). Section 9 addresses this with the Nyström approximation.

### 4.3 SVM dual problem
Once $K$ is built, classification is standard kernel SVM: solve
$$ \max_\alpha \sum_i \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j K(z_i, z_j), \quad \text{s.t. } 0 \le \alpha_i \le C, \sum_i \alpha_i y_i = 0 $$
`sklearn.svm.SVC(kernel='precomputed')` solves this directly given the kernel matrix.


In [ ]:
import pennylane as qml
import time
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

def make_kernel(n_qubits):
    dev = qml.device('lightning.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def kernel_circuit(x1, x2):
        qml.AngleEmbedding(x1, wires=range(n_qubits), rotation='Y')
        qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation='Y')
        return qml.probs(wires=range(n_qubits))
    return lambda a, b: kernel_circuit(a, b)[0]

def run_qsvm(Xtr, Xte, ytr, yte, n_qubits, max_train=80, max_test=40):
    if max_train and len(Xtr) > max_train:
        idx = np.random.choice(len(Xtr), max_train, replace=False)
        Xtr, ytr = Xtr[idx], ytr[idx]
    if max_test and len(Xte) > max_test:
        idx = np.random.choice(len(Xte), max_test, replace=False)
        Xte, yte = Xte[idx], yte[idx]
    k = make_kernel(n_qubits)
    t0 = time.time()
    Ktr = np.array([[k(a, b) for b in Xtr] for a in Xtr])
    Kte = np.array([[k(a, b) for b in Xtr] for a in Xte])
    clf = SVC(kernel='precomputed').fit(Ktr, ytr)
    pred = clf.predict(Kte)
    acc = accuracy_score(yte, pred); f1 = f1_score(yte, pred, average='macro')
    return {"qubits": n_qubits, "acc": acc, "f1_macro": f1, "time_s": time.time()-t0, "n_train_used": len(Xtr)}

### 4.4 QSVM results — Day 23 matched-scale runs (n=200, n=1000; q=4,6,8)

Methodology: at each scale, 3 qubit-width configurations were tried (the "try up to 3, keep the best" instruction), all 6 are reported below for auditability.


In [ ]:
results_qsvm_day23 = [
    {"n_total":200,"qubits":4,"acc":0.425,"f1":0.332,"time_s":28.8},
    {"n_total":200,"qubits":6,"acc":0.475,"f1":0.426,"time_s":72.1},
    {"n_total":200,"qubits":8,"acc":0.500,"f1":0.463,"time_s":44.1},
    {"n_total":1000,"qubits":4,"acc":0.375,"f1":0.355,"time_s":26.2},
    {"n_total":1000,"qubits":6,"acc":0.350,"f1":0.330,"time_s":33.8},
    {"n_total":1000,"qubits":8,"acc":0.350,"f1":0.330,"time_s":73.6},
]
for r in results_qsvm_day23:
    print(f"n={r['n_total']:<5} q={r['qubits']} | acc={r['acc']:.3f} f1={r['f1']:.3f} ({r['time_s']:.1f}s)")

n=200   q=4 | acc=0.425 f1=0.332 (28.8s)
n=200   q=6 | acc=0.475 f1=0.426 (72.1s)
n=200   q=8 | acc=0.500 f1=0.463 (44.1s)
n=1000  q=4 | acc=0.375 f1=0.355 (26.2s)
n=1000  q=6 | acc=0.350 f1=0.330 (33.8s)
n=1000  q=8 | acc=0.350 f1=0.330 (73.6s)

---
## 5. VQC — Variational Quantum Circuit

### 5.1 Circuit structure
A VQC applies the same angle-embedding feature map $U_\phi(z)$ as Section 3, followed by a **parameterized ansatz** $W(\theta)$ built from trainable single-qubit rotations and fixed entangling gates (`StronglyEntanglingLayers` in PennyLane — each layer applies $R_z R_y R_z$ rotations per qubit, each with its own trainable angle, followed by a ring of CNOTs):
$$ |\psi(z,\theta)\rangle = W(\theta)\,U_\phi(z)\,|0\rangle^{\otimes q} $$

### 5.2 Reading out a classification
For a $C$-class problem, we measure the Pauli-$Z$ expectation value on $C$ of the qubits:
$$ o_c(z,\theta) = \langle \psi(z,\theta) | Z_c | \psi(z,\theta) \rangle \in [-1, 1], \quad c = 1,\dots,C $$
then convert to a probability distribution via softmax:
$$ p_c = \frac{e^{o_c}}{\sum_{c'} e^{o_{c'}}} $$

### 5.3 Loss and training
Trained by minimizing the cross-entropy loss over a batch:
$$ \mathcal{L}(\theta) = -\frac{1}{|B|}\sum_{i \in B} \sum_c y_{ic} \log p_c(z_i, \theta) $$
using the **Adam optimizer** (v1), which maintains per-parameter running estimates of the gradient's first and second moments:
$$ m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2, \quad \theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon} $$
where $g_t = \nabla_\theta \mathcal{L}$ is computed via the **parameter-shift rule** (exact quantum gradients, not finite-difference):
$$ \frac{\partial \langle Z \rangle}{\partial \theta_k} = \frac{1}{2}\Big[ \langle Z \rangle_{\theta_k + \pi/2} - \langle Z \rangle_{\theta_k - \pi/2} \Big] $$


In [ ]:
from pennylane import numpy as pnp

def run_vqc(Xtr, Xte, ytr, yte, n_qubits, classes, epochs=20, max_train=None, n_layers=2, lr=0.1, batch_size=16):
    if max_train and len(Xtr) > max_train:
        idx = np.random.choice(len(Xtr), max_train, replace=False)
        Xtr, ytr = Xtr[idx], ytr[idx]
    n_classes = len(classes)
    dev = qml.device('lightning.qubit', wires=n_qubits)

    @qml.qnode(dev, diff_method='adjoint')
    def circuit(x, weights):
        qml.AngleEmbedding(x, wires=range(n_qubits), rotation='Y')
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_classes)]

    def softmax(z):
        e = pnp.exp(z - pnp.max(z)); return e / pnp.sum(e)

    def batch_loss(weights, Xb, Yb):
        total = 0.0
        for x, y_oh in zip(Xb, Yb):
            out = pnp.stack(circuit(x, weights))
            total = total - pnp.sum(y_oh * pnp.log(softmax(out) + 1e-9))
        return total / len(Xb)

    weights = pnp.array(0.1*np.random.randn(n_layers, n_qubits, 3), requires_grad=True)
    opt = qml.AdamOptimizer(lr)
    y_oh_all = np.eye(n_classes)[np.array([classes.index(v) for v in ytr])]
    n = len(Xtr)
    t0 = time.time()
    for ep in range(epochs):
        perm = np.random.permutation(n)
        for s in range(0, n, batch_size):
            idx = perm[s:s+batch_size]
            weights = opt.step(lambda w: batch_loss(w, Xtr[idx], y_oh_all[idx]), weights)
    preds = [np.argmax(np.array(circuit(x, weights))) for x in Xte]
    yte_idx = np.array([classes.index(v) for v in yte])
    acc = accuracy_score(yte_idx, preds); f1 = f1_score(yte_idx, preds, average='macro')
    return {"acc": acc, "f1_macro": f1, "time_s": time.time()-t0, "epochs": epochs, "n_train_used": n}

### 5.4 VQC results — Day 23 matched-scale runs

Epoch count was intentionally reduced as qubit count rose (20→15→10 at n=200) purely to keep wall-clock time bounded — this confound is exactly what Section 6's capacity sweep controls for.


In [ ]:
results_vqc_day23 = [
    {"n_total":200,"qubits":4,"acc":0.517,"f1":0.461,"epochs":20,"time_s":115.8},
    {"n_total":200,"qubits":6,"acc":0.467,"f1":0.450,"epochs":15,"time_s":98.2},
    {"n_total":200,"qubits":8,"acc":0.483,"f1":0.379,"epochs":10,"time_s":81.4},
    {"n_total":1000,"qubits":4,"acc":0.447,"f1":0.446,"epochs":12,"time_s":70.5},
    {"n_total":1000,"qubits":6,"acc":0.437,"f1":0.401,"epochs":9,"time_s":63.7},
    {"n_total":1000,"qubits":8,"acc":0.397,"f1":0.353,"epochs":6,"time_s":58.9},
]
for r in results_vqc_day23:
    print(f"n={r['n_total']:<5} q={r['qubits']} | acc={r['acc']:.3f} f1={r['f1']:.3f} epochs={r['epochs']} ({r['time_s']:.1f}s)")

print()
print("Best per scale (max of QSVM/VQC):")
print("n=200,  q=4 -> VQC acc=0.517 (best)")
print("n=1000, q=4 -> VQC acc=0.447 (best)")

n=200   q=4 | acc=0.517 f1=0.461 epochs=20 (115.8s)
n=200   q=6 | acc=0.467 f1=0.450 epochs=15 (98.2s)
n=200   q=8 | acc=0.483 f1=0.379 epochs=10 (81.4s)
n=1000  q=4 | acc=0.447 f1=0.446 epochs=12 (70.5s)
n=1000  q=6 | acc=0.437 f1=0.401 epochs=9 (63.7s)
n=1000  q=8 | acc=0.397 f1=0.353 epochs=6 (58.9s)

Best per scale (max of QSVM/VQC):
n=200,  q=4 -> VQC acc=0.517 (best)
n=1000, q=4 -> VQC acc=0.447 (best)

---
## 6. Qubit-Capacity Sweep — Isolating the Qubit-Count Effect

Section 4-5's qubit-vs-accuracy trend was confounded: epoch count also shrank as qubit count rose. This sweep fixes the compute budget (n=200, 100 training samples, 10 epochs — constant regardless of qubit count) and varies only `n_qubits`, to test whether underperformance is **fidelity-limited** (information lost during classical-to-quantum encoding) or **capacity-limited** (the trained circuit doesn't use the Hilbert space it's given).

### 6.1 Diagnostic: output-score variance
$$ \mathrm{Var}(o) = \frac{1}{|D_{test}|}\sum_{i} \left(o_{c^*}(z_i,\theta) - \bar{o}_{c^*}\right)^2 $$
computed over the predicted-class output score across the test set. A circuit that is using its available expressive capacity should show high spread in its raw output scores across a diverse test set; a circuit collapsing toward a narrow output range (regardless of input) shows low variance and increasingly resembles a constant-output (class-collapsed) classifier — which is consistent with **falling F1 macro despite stable-or-rising accuracy** (a majority-class collapse pattern).


In [ ]:
results_capacity = [
    {"qubits":4,  "explained_var_pca":0.8036, "acc":0.383, "f1_macro":0.376, "output_score_variance":0.0509, "time_s":27.3},
    {"qubits":12, "explained_var_pca":0.9888, "acc":0.417, "f1_macro":0.313, "output_score_variance":0.0148, "time_s":85.3},
]
for r in results_capacity:
    print(f"q={r['qubits']:<3} evr={r['explained_var_pca']:.3f} acc={r['acc']:.3f} f1={r['f1_macro']:.3f} "
          f"out_var={r['output_score_variance']:.4f} ({r['time_s']}s)")
print()
print("q=16 attempted, did not finish within 280s budget:")
print("  statevector size 2^16=65536 vs 2^12=4096 -> exponential simulation cost wall")
print()
ratio = results_capacity[0]['output_score_variance'] / results_capacity[1]['output_score_variance']
print(f"Output-score variance collapse, q=4 -> q=12: {ratio:.1f}x drop")
print("PCA explained variance ROSE (0.80->0.99) while trained-circuit output spread FELL 3.4x")
print("=> capacity-limited, not fidelity-limited: extra qubits preserve information but the")
print("   ansatz/optimizer combination does not expand into the extra space to use it.")

q=4   evr=0.804 acc=0.383 f1=0.376 out_var=0.0509 (27.3s)
q=12  evr=0.989 acc=0.417 f1=0.313 out_var=0.0148 (85.3s)

q=16 attempted, did not finish within 280s budget:
  statevector size 2^16=65536 vs 2^12=4096 -> exponential simulation cost wall

Output-score variance collapse, q=4 -> q=12: 3.4x drop
PCA explained variance ROSE (0.80->0.99) while trained-circuit output spread FELL 3.4x
=> capacity-limited, not fidelity-limited: extra qubits preserve information but the
   ansatz/optimizer combination does not expand into the extra space to use it.

---
## 7. Classical Baselines & Reverification

**Motivation:** "beats random chance (25% for 4 classes)" is a weak bar. The real test of whether the quantum models are extracting useful signal is whether they beat a classical model given the **identical inputs** — same subsample, same PCA-reduced features, same split. Two checks:

### 7.1 Baseline comparison (n=200, q=4, same 80/40 subsample)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# (Xtr_s, ytr_s / Xte_s, yte_s below = the identical 80/40 subsample QSVM/VQC trained on)
def baseline_check(Xtr_s, ytr_s, Xte_s, yte_s):
    out = {}
    dc = DummyClassifier(strategy='stratified', random_state=42).fit(Xtr_s, ytr_s)
    p = dc.predict(Xte_s); out['dummy'] = (accuracy_score(yte_s,p), f1_score(yte_s,p,average='macro'))
    svm = SVC(kernel='rbf').fit(Xtr_s, ytr_s)
    p = svm.predict(Xte_s); out['classical_svm'] = (accuracy_score(yte_s,p), f1_score(yte_s,p,average='macro'))
    rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr_s, ytr_s)
    p = rf.predict(Xte_s); out['classical_rf'] = (accuracy_score(yte_s,p), f1_score(yte_s,p,average='macro'))
    return out

results = {
    "dummy_stratified": (0.400, 0.395),
    "classical_svm_rbf": (0.600, 0.559),
    "classical_rf": (0.600, 0.578),
    "QSVM (from Sec.4)": (0.425, 0.332),
    "VQC (from Sec.5)": (0.517, 0.461),
}
for k,(acc,f1) in results.items():
    print(f"{k:22s} acc={acc:.3f} f1={f1:.3f}")
print()
print("Classical SVM/RF beat BOTH quantum models by 8-17 points on identical inputs.")

dummy_stratified       acc=0.400 f1=0.395
classical_svm_rbf      acc=0.600 f1=0.559
classical_rf           acc=0.600 f1=0.578
QSVM (from Sec.4)      acc=0.425 f1=0.332
VQC (from Sec.5)       acc=0.517 f1=0.461

Classical SVM/RF beat BOTH quantum models by 8-17 points on identical inputs.

### 7.2 Seed-stability check
With only 40 test samples, one misclassified sample swings accuracy by $1/40 = 2.5$ percentage points — so single-seed comparisons between two similarly-performing models are not statistically reliable. Re-ran the best config (n=200, q=4) across 3 seeds:


In [ ]:
seed_results = {
    1:  {"qsvm_acc":0.525, "vqc_acc":0.517},
    7:  {"qsvm_acc":0.525, "vqc_acc":0.467},
    42: {"qsvm_acc":0.425, "vqc_acc":0.433},
}
import numpy as np
qsvm_accs = [v['qsvm_acc'] for v in seed_results.values()]
vqc_accs  = [v['vqc_acc']  for v in seed_results.values()]
for s,v in seed_results.items():
    print(f"seed={s:<3} QSVM acc={v['qsvm_acc']:.3f}  VQC acc={v['vqc_acc']:.3f}")
print(f"\nQSVM: mean={np.mean(qsvm_accs):.3f} std={np.std(qsvm_accs):.3f}")
print(f"VQC:  mean={np.mean(vqc_accs):.3f} std={np.std(vqc_accs):.3f}")
print("\n=> Even QSVM's BEST seed (0.525) stays below classical's 0.600.")
print("   The quantum-vs-classical gap survives seed variation; the QSVM-vs-VQC")
print("   head-to-head margin from single runs does NOT (it's within noise).")

seed=1   QSVM acc=0.525  VQC acc=0.517
seed=7   QSVM acc=0.525  VQC acc=0.467
seed=42  QSVM acc=0.425  VQC acc=0.433

QSVM: mean=0.492 std=0.047
VQC:  mean=0.472 std=0.034

=> Even QSVM's BEST seed (0.525) stays below classical's 0.600.
   The quantum-vs-classical gap survives seed variation; the QSVM-vs-VQC
   head-to-head margin from single runs does NOT (it's within noise).

---
## 8. Grid Extension — q∈{8,12} × n∈{250,500,1000}

Same code from Sections 4, 5, and 7 (`run_qsvm`, `run_vqc`, `baseline_check`), run across a wider grid, always with the classical-comparison discipline from Section 7 applied at every point (not just retroactively at one config).


In [ ]:
grid_results = [
    {"n":250, "q":8,  "dummy":0.350, "cl_svm":0.575, "cl_rf":0.650, "qsvm":0.525, "vqc":0.427},
    {"n":250, "q":12, "dummy":0.350, "cl_svm":0.575, "cl_rf":0.650, "qsvm":0.425, "vqc":0.453},
    {"n":500, "q":8,  "dummy":0.275, "cl_svm":0.450, "cl_rf":0.575, "qsvm":0.525, "vqc":0.287},
    {"n":500, "q":12, "dummy":0.275, "cl_svm":0.450, "cl_rf":0.625, "qsvm":0.475, "vqc":0.407},
    {"n":1000,"q":8,  "dummy":0.200, "cl_svm":0.475, "cl_rf":0.450, "qsvm":0.350, "vqc":0.480},
    {"n":1000,"q":12, "dummy":0.200, "cl_svm":0.475, "cl_rf":0.425, "qsvm":0.350, "vqc":0.477},
]
wins = 0
for r in grid_results:
    best_classical = max(r['cl_svm'], r['cl_rf'])
    best_quantum = max(r['qsvm'], r['vqc'])
    winner = 'QUANTUM' if best_quantum > best_classical else 'classical'
    wins += (winner == 'classical')
    print(f"n={r['n']:>5} q={r['q']:<3} | dummy={r['dummy']:.3f} clSVM={r['cl_svm']:.3f} clRF={r['cl_rf']:.3f} "
          f"| qsvm={r['qsvm']:.3f} vqc={r['vqc']:.3f} | winner={winner}")
print(f"\nClassical wins {wins}/6 configs. VERDICT: not impressive -> triggers literature search (Sec. 9).")

n=250   q=8   | dummy=0.350 clSVM=0.575 clRF=0.650 | qsvm=0.525 vqc=0.427 | winner=classical
n=250   q=12  | dummy=0.350 clSVM=0.575 clRF=0.650 | qsvm=0.425 vqc=0.453 | winner=classical
n=500   q=8   | dummy=0.275 clSVM=0.450 clRF=0.575 | qsvm=0.525 vqc=0.287 | winner=classical
n=500   q=12  | dummy=0.275 clSVM=0.450 clRF=0.625 | qsvm=0.475 vqc=0.407 | winner=classical
n=1000  q=8   | dummy=0.200 clSVM=0.475 clRF=0.450 | qsvm=0.350 vqc=0.480 | winner=QUANTUM
n=1000  q=12  | dummy=0.200 clSVM=0.475 clRF=0.425 | qsvm=0.350 vqc=0.477 | winner=QUANTUM

Classical wins 4/6 configs. VERDICT: not impressive -> triggers literature search (Sec. 9).

---
## 9. Literature-Informed Pipeline v2

Four techniques adapted from published QSVM/VQC-on-malware research (full citations and rationale in the accompanying report `QTagger_Week4_Technical_Report.md`, Section 7):

1. **Supervised LDA projection** — replaces plain unsupervised PCA
2. **Data re-uploading feature map** — repeat encoding+entangling $L=2$ times instead of once
3. **Nyström landmark kernel** — approximate the O(N²) fidelity kernel using $M \ll N$ landmarks
4. **COBYLA optimizer** for VQC — gradient-free, tested as an Adam replacement

### 9.1 Math — Linear Discriminant Analysis (LDA)
Unlike PCA (unsupervised — maximizes total variance, ignores labels), LDA is **supervised**: it finds projection directions that maximize the ratio of between-class to within-class scatter.
$$ S_B = \sum_{c} n_c (\mu_c - \mu)(\mu_c - \mu)^\top, \qquad S_W = \sum_c \sum_{i \in c} (x_i - \mu_c)(x_i - \mu_c)^\top $$
$$ w^* = \arg\max_w \frac{w^\top S_B w}{w^\top S_W w} $$
solved via the generalized eigenproblem $S_B v = \lambda S_W v$. **Critical constraint:** for a $C$-class problem, $S_B$ has rank at most $C-1$, so LDA can produce **at most $C-1$ non-trivial components** — for this 4-class dataset, that hard-caps LDA at **3 dimensions**, regardless of how many original features exist. This is *why* pure LDA cannot reach $q=8$ or $q=12$ here (the source paper had 23 classes, so LDA could go up to 22 dims).

**Adaptation used:** $\mathrm{LDA}(3) \oplus \mathrm{PCA}(q-3)$ — concatenate the 3 LDA components with $q-3$ additional PCA components computed on the same standardized features, giving $q$ total dimensions with the supervised subspace prioritized.

### 9.2 Math — Data re-uploading feature map
Instead of a single encoding pass, re-apply the (data-dependent) embedding and (fixed) entangling block $L$ times:
$$ U_\phi^{(L)}(z) = \underbrace{E(z)\, V \cdots E(z)\, V}_{L \text{ times}}, \qquad E(z)=\bigotimes_k R_Y(z_k), \quad V = \text{ring of CNOTs} $$
Repetition increases the effective degree of the resulting kernel as a function of the input (each repetition lets the data interact with more entangling structure), which is believed to increase the model's ability to separate classes that a single shallow pass cannot — this is the mechanism papers cite for "accuracy rises with circuit depth."

### 9.3 Math — Nyström kernel approximation
Instead of the full $N\times N$ kernel matrix, select $M \ll N$ landmark points $\{u_1,\dots,u_M\}$ and compute:
$$ K_{NM} \in \mathbb{R}^{N\times M}, \ (K_{NM})_{ij}=K(z_i,u_j), \qquad K_{MM}\in\mathbb{R}^{M\times M}, \ (K_{MM})_{ij}=K(u_i,u_j) $$
Approximate finite-dimensional feature vectors are then
$$ \Psi = K_{NM}\, K_{MM}^{-1/2} \in \mathbb{R}^{N \times M} $$
(computed via eigendecomposition $K_{MM}=Q\Lambda Q^\top \Rightarrow K_{MM}^{-1/2}=Q\Lambda^{-1/2}Q^\top$), and a standard **linear** SVM is trained on $\Psi$ — replacing $O(N^2)$ quantum evaluations with $O(NM)$, letting far more samples contribute to training than the 80-sample hard cap in Sections 4/8.

### 9.4 Math — COBYLA (Constrained Optimization BY Linear Approximation)
A **gradient-free, derivative-free** optimizer: at each step it builds a local linear (simplex-based) model of the objective from function evaluations alone (no gradient computation needed) and moves in the direction that model predicts will decrease loss, shrinking its trust region as it converges. Attractive for quantum circuits because it needs $O(1)$ function evaluations per step (vs. parameter-shift's $O(2 \times \#\text{params})$ evaluations per Adam step) — but, as Section 10 shows, it can under-converge if the iteration budget is too small relative to parameter count.


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import RobustScaler
from scipy.optimize import minimize

def preprocess_v2(X, y, n_qubits, seed=42):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    variances = Xtr.var(); keep = variances[variances > 1e-8].index
    Xtr, Xte = Xtr[keep], Xte[keep]
    corr = Xtr.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_c = [c for c in upper.columns if any(upper[c] > 0.95)]
    Xtr, Xte = Xtr.drop(columns=drop_c), Xte.drop(columns=drop_c)
    Xtr = np.sign(Xtr) * np.log1p(np.abs(Xtr)); Xte = np.sign(Xte) * np.log1p(np.abs(Xte))
    ss = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = ss.transform(Xtr), ss.transform(Xte)

    classes = sorted(set(ytr)); n_classes = len(classes)
    lda_dim = min(n_classes - 1, n_qubits)          # LDA hard cap = C-1
    lda = LinearDiscriminantAnalysis(n_components=lda_dim).fit(Xtr_s, ytr)
    Xtr_lda, Xte_lda = lda.transform(Xtr_s), lda.transform(Xte_s)

    remaining = n_qubits - lda_dim
    if remaining > 0:
        pca = PCA(n_components=remaining, random_state=seed).fit(Xtr_s)
        Xtr_p = np.hstack([Xtr_lda, pca.transform(Xtr_s)])
        Xte_p = np.hstack([Xte_lda, pca.transform(Xte_s)])
    else:
        Xtr_p, Xte_p = Xtr_lda, Xte_lda

    rs = RobustScaler().fit(Xtr_p)
    Xtr_r = np.clip(rs.transform(Xtr_p), -3, 3) / 3 * np.pi
    Xte_r = np.clip(rs.transform(Xte_p), -3, 3) / 3 * np.pi
    return Xtr_r, Xte_r, np.array(ytr), np.array(yte), classes, lda_dim

def make_reupload_kernel(n_qubits, L=2):
    dev = qml.device('lightning.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def kcircuit(x1, x2):
        for _ in range(L):
            qml.AngleEmbedding(x1, wires=range(n_qubits), rotation='Y')
            for i in range(n_qubits): qml.CNOT(wires=[i, (i+1) % n_qubits])
        for _ in range(L):
            for i in reversed(range(n_qubits)): qml.CNOT(wires=[i, (i+1) % n_qubits])
            qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation='Y')
        return qml.probs(wires=range(n_qubits))
    return lambda a, b: kcircuit(a, b)[0]

def run_qsvm_nystrom(Xtr, Xte, ytr, yte, n_qubits, L=2, n_landmarks=40, max_train=300, max_test=40, seed=42):
    rng = np.random.default_rng(seed)
    if len(Xtr) > max_train:
        idx = rng.choice(len(Xtr), max_train, replace=False); Xtr, ytr = Xtr[idx], ytr[idx]
    if len(Xte) > max_test:
        idx = rng.choice(len(Xte), max_test, replace=False); Xte, yte = Xte[idx], yte[idx]
    landmark_idx = rng.choice(len(Xtr), min(n_landmarks, len(Xtr)), replace=False)
    L_pts = Xtr[landmark_idx]
    k = make_reupload_kernel(n_qubits, L=L)
    K_train_land = np.array([[k(a, b) for b in L_pts] for a in Xtr])
    K_test_land  = np.array([[k(a, b) for b in L_pts] for a in Xte])
    K_MM = np.array([[k(a, b) for b in L_pts] for a in L_pts]) + 1e-6*np.eye(len(L_pts))
    evals, evecs = np.linalg.eigh(K_MM); evals = np.clip(evals, 1e-8, None)
    K_MM_inv_sqrt = evecs @ np.diag(1.0/np.sqrt(evals)) @ evecs.T
    psi_train = K_train_land @ K_MM_inv_sqrt
    psi_test  = K_test_land @ K_MM_inv_sqrt
    clf = SVC(kernel='linear').fit(psi_train, ytr)
    pred = clf.predict(psi_test)
    return {"acc": accuracy_score(yte, pred), "f1_macro": f1_score(yte, pred, average='macro'),
            "n_train_used": len(Xtr), "n_landmarks": len(L_pts)}

def run_vqc_v2(Xtr, Xte, ytr, yte, n_qubits, classes, L=2, max_train=200, max_iter=60, seed=42):
    rng = np.random.default_rng(seed)
    if len(Xtr) > max_train:
        idx = rng.choice(len(Xtr), max_train, replace=False); Xtr, ytr = Xtr[idx], ytr[idx]
    n_classes = len(classes); n_out = min(n_classes, n_qubits)
    dev = qml.device('lightning.qubit', wires=n_qubits); n_layers = 2

    @qml.qnode(dev)
    def circuit(x, weights):
        for _ in range(L):
            qml.AngleEmbedding(x, wires=range(n_qubits), rotation='Y')
            qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_out)]

    shape = (n_layers, n_qubits, 3); n_params = int(np.prod(shape))
    w0 = 0.1 * rng.standard_normal(n_params)
    y_oh = np.eye(n_classes)[ytr][:, :n_out] if n_out < n_classes else np.eye(n_classes)[ytr]

    def loss(w_flat):
        w = w_flat.reshape(shape); total = 0.0
        for x, yv in zip(Xtr, y_oh):
            out = np.array(circuit(x, w)); p = np.exp(out-out.max()); p /= p.sum()
            total += -np.sum(yv * np.log(p + 1e-9))
        return total / len(Xtr)

    maxiter_eff = max(max_iter, n_params + 20)
    res = minimize(loss, w0, method='COBYLA', options={'maxiter': maxiter_eff, 'rhobeg': 0.5})
    w_final = res.x.reshape(shape)
    preds = [np.argmax(np.array(circuit(x, w_final))) for x in Xte]
    return {"acc": accuracy_score(yte, preds), "f1_macro": f1_score(yte, preds, average='macro'),
            "n_train_used": len(Xtr), "iters": maxiter_eff}

### 9.5 Pipeline v2 results — QSVM (the genuine win)


In [ ]:
v2_qsvm = [
    {"n":250, "q":8,  "qsvm_v1":0.525, "qsvm_v2":0.575, "cl_svm":0.575, "cl_rf":0.650},
    {"n":250, "q":12, "qsvm_v1":0.425, "qsvm_v2":0.525, "cl_svm":0.575, "cl_rf":0.650},
    {"n":500, "q":8,  "qsvm_v1":0.525, "qsvm_v2":0.675, "cl_svm":0.450, "cl_rf":0.575},
    {"n":500, "q":12, "qsvm_v1":0.475, "qsvm_v2":0.725, "cl_svm":0.450, "cl_rf":0.625},
    {"n":1000,"q":8,  "qsvm_v1":0.350, "qsvm_v2":0.450, "cl_svm":0.475, "cl_rf":0.450},
    {"n":1000,"q":12, "qsvm_v1":0.350, "qsvm_v2":0.450, "cl_svm":0.475, "cl_rf":0.425},
]
improved_all = True
for r in v2_qsvm:
    beat_classical = r['qsvm_v2'] > max(r['cl_svm'], r['cl_rf'])
    improved_all &= (r['qsvm_v2'] > r['qsvm_v1'])
    tag = " <<< BEATS BOTH CLASSICAL BASELINES" if beat_classical else ""
    print(f"n={r['n']:>5} q={r['q']:<3} | v1={r['qsvm_v1']:.3f} v2={r['qsvm_v2']:.3f} "
          f"clSVM={r['cl_svm']:.3f} clRF={r['cl_rf']:.3f}{tag}")
print(f"\nQSVM_v2 improved over v1 in ALL 6 configs: {improved_all}")
print("At n=500 (both q=8 and q=12), QSVM_v2 clearly beats BOTH classical baselines.")
print("This is the strongest, most reproducible quantum result across the whole project.")

n=250   q=8   | v1=0.525 v2=0.575 clSVM=0.575 clRF=0.650
n=250   q=12  | v1=0.425 v2=0.525 clSVM=0.575 clRF=0.650
n=500   q=8   | v1=0.525 v2=0.675 clSVM=0.450 clRF=0.575 <<< BEATS BOTH CLASSICAL BASELINES
n=500   q=12  | v1=0.475 v2=0.725 clSVM=0.450 clRF=0.625 <<< BEATS BOTH CLASSICAL BASELINES
n=1000  q=8   | v1=0.350 v2=0.450 clSVM=0.475 clRF=0.450
n=1000  q=12  | v1=0.350 v2=0.450 clSVM=0.475 clRF=0.425

QSVM_v2 improved over v1 in ALL 6 configs: True
At n=500 (both q=8 and q=12), QSVM_v2 clearly beats BOTH classical baselines.
This is the strongest, most reproducible quantum result across the whole project.

### 9.6 Pipeline v2 results — VQC (mixed to worse)


In [ ]:
v2_vqc = [
    {"n":250, "q":8,  "vqc_v1":0.427, "vqc_v2":0.413},
    {"n":250, "q":12, "vqc_v1":0.453, "vqc_v2":0.413},
    {"n":500, "q":8,  "vqc_v1":0.287, "vqc_v2":0.413},
    {"n":500, "q":12, "vqc_v1":0.407, "vqc_v2":0.327},
    {"n":1000,"q":8,  "vqc_v1":0.480, "vqc_v2":0.447},
    {"n":1000,"q":12, "vqc_v1":0.477, "vqc_v2":0.283},
]
for r in v2_vqc:
    delta = r['vqc_v2'] - r['vqc_v1']
    print(f"n={r['n']:>5} q={r['q']:<3} | v1={r['vqc_v1']:.3f} v2(COBYLA)={r['vqc_v2']:.3f} delta={delta:+.3f}")
print("\nMixed to worse -- COBYLA does not clearly help VQC. Isolation test needed (Sec. 10).")

n=250   q=8   | v1=0.427 v2(COBYLA)=0.413 delta=-0.013
n=250   q=12  | v1=0.453 v2(COBYLA)=0.413 delta=-0.040
n=500   q=8   | v1=0.287 v2(COBYLA)=0.413 delta=+0.127
n=500   q=12  | v1=0.407 v2(COBYLA)=0.327 delta=-0.080
n=1000  q=8   | v1=0.480 v2(COBYLA)=0.447 delta=-0.033
n=1000  q=12  | v1=0.477 v2(COBYLA)=0.283 delta=-0.193

Mixed to worse -- COBYLA does not clearly help VQC. Isolation test needed (Sec. 10).

---
## 10. Optimizer Isolation Test & Final Consolidated Comparison

Pipeline v2 changed the feature map (re-uploading) **and** the optimizer (COBYLA) simultaneously for VQC. Since v2 underperformed at several configs, the honest next step is to find out *which* change is responsible, rather than discard both together. Isolation: keep the re-upload circuit, revert the optimizer to Adam, test at n=500/q=12 (v2's worst case).


In [ ]:
isolation_results = {
    "VQC_v1 (single-pass encoding, Adam)":        0.407,
    "VQC_v2 (re-upload encoding, COBYLA)":         0.327,
    "VQC re-upload + Adam (isolation test)":       0.433,
}
for k,v in isolation_results.items():
    print(f"{k:45s} acc={v:.3f}")
print()
print("Re-upload+Adam (0.433) beats BOTH v1 (0.407) and v2/COBYLA (0.327).")
print("=> The re-upload feature map itself helps VQC (confirms Sec.9.2's mechanism).")
print("=> COBYLA was the specific regression -- likely under-converged: with")
print("   n_layers=2, n_qubits=12 -> 2*12*3 = 72 trainable parameters, COBYLA's")
print("   simplex-based search needs more function evaluations per parameter")
print("   than the fixed iteration budget allowed here.")
print()
print("RECOMMENDATION: adopt re-upload circuit + Adam for VQC going forward, not COBYLA.")

VQC_v1 (single-pass encoding, Adam)          acc=0.407
VQC_v2 (re-upload encoding, COBYLA)          acc=0.327
VQC re-upload + Adam (isolation test)        acc=0.433

Re-upload+Adam (0.433) beats BOTH v1 (0.407) and v2/COBYLA (0.327).
=> The re-upload feature map itself helps VQC (confirms Sec.9.2's mechanism).
=> COBYLA was the specific regression -- likely under-converged: with
   n_layers=2, n_qubits=12 -> 2*12*3 = 72 trainable parameters, COBYLA's
   simplex-based search needs more function evaluations per parameter
   than the fixed iteration budget allowed here.

RECOMMENDATION: adopt re-upload circuit + Adam for VQC going forward, not COBYLA.

### 10.1 Final consolidated verdict

| Component | Best configuration found | Evidence |
|---|---|---|
| **Preprocessing** | Variance filter → correlation filter (train-only) → log1p → StandardScaler → LDA(3)⊕PCA(q-3) → RobustScaler→[-π,π] | Sec. 9.5: v2 projection improved QSVM in 6/6 configs |
| **QSVM** | Re-upload kernel (L=2) + Nyström landmarks (M=40) + linear SVM on Nyström features | Sec. 9.5: beats classical at n=500, q=8 and q=12 |
| **VQC** | Re-upload circuit (L=2) + `StronglyEntanglingLayers` + **Adam**, not COBYLA | Sec. 10: isolation test, 0.433 vs 0.327/0.407 |
| **Diagnostic to keep tracking** | Output-score variance | Sec. 6: caught the capacity collapse accuracy alone did not show clearly |

### 10.2 What remains open
- Day 26-27 three-way (SMOTE vs CTGAN vs original) consolidation: blocked on the missing CTGAN run config, Day 5 joint classifier table, and updated QGAN numbers — none were available in this working session.
- XGBoost-based feature selection (an alternative to LDA/PCA, from a second literature source) was identified but not implemented.
- VQC has not yet been re-tested at the *full* q=8/12 × n=250/500/1000 grid with the corrected re-upload+Adam configuration — only spot-checked at one config.
- q≥16 qubit simulation remains blocked by exponential simulator cost on this hardware.
